## STEP 1 : Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import joblib
import json

## STEP 2 : Load Dataset

In [2]:
df = pd.read_csv(r"E:\Data Science Projects live\house_price_prediction_app\House_data.csv")
df.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


## STEP 3: Select Required Columns

In [3]:
features = [
    "bedrooms",
    "bathrooms",
    "sqft_living",
    "floors",
    "grade",
    "lat",
    "long"
]

target = "price"

df_model = df[features + [target]]
df_model.head()


,bedrooms,bathrooms,sqft_living,floors,grade,lat,long,price
0,3,1.00,1180,1.0,7,47.5112,-122.257,221900.0
1,3,2.25,2570,2.0,7,47.7210,-122.319,538000.0
2,2,1.00,770,1.0,6,47.7379,-122.233,180000.0
3,4,3.00,1960,1.0,7,47.5208,-122.393,604000.0
4,3,2.00,1680,1.0,8,47.6168,-122.045,510000.0


## STEP 4: Split X & y

In [4]:
X = df_model[features]
y = df_model[target]


## STEP 5: Train–Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


## STEP 6: Feature Scaling

In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## STEP 7: Model Training (Random Forest)

In [7]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=120,      # ↓ from 150
    max_depth=10,          # controls tree size
    min_samples_leaf=5,    # reduces overfitting + size
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_scaled, y_train)


RandomForestRegressor(max_depth=10, min_samples_leaf=5, n_estimators=120,
                      n_jobs=-1, random_state=42)

## STEP 8: Model Evaluation

In [8]:
y_pred = rf.predict(X_test_scaled)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("R2 Score :", r2)
print("MAE      :", mae)
print("RMSE     :", rmse)


R2 Score : 0.81464357997084
MAE      : 84887.66036589451
RMSE     : 167494.9728959414


## STEP 9: Manual Prediction Test

In [9]:
import warnings
warnings.filterwarnings("ignore")

In [10]:
sample_1 = np.array([[3, 2, 1500, 1, 7, 47.6, -122.3]])
sample_2 = np.array([[4, 3, 2500, 2, 9, 47.6, -122.3]])

sample_1_scaled = scaler.transform(sample_1)
sample_2_scaled = scaler.transform(sample_2)

print("Prediction 1:", rf.predict(sample_1_scaled))
print("Prediction 2:", rf.predict(sample_2_scaled))


Prediction 1: [536209.85829119]
Prediction 2: [1030043.47439366]


## STEP 10: Save Model, Scaler & Feature Names

In [11]:
import joblib
import os
import json

# Save compressed model
joblib.dump(
    rf,
    "house_price_model.pkl",
    compress=5   # best balance
)

# Save scaler
joblib.dump(
    scaler,
    "scaler.pkl",
    compress=3
)

# Save feature names
with open("feature_names.json", "w") as f:
    json.dump(features, f)

# Check file sizes
print("Model size (MB):",
      os.path.getsize("house_price_model.pkl") / (1024*1024))

print("Scaler size (MB):",
      os.path.getsize("scaler.pkl") / (1024*1024))


Model size (MB): 2.7216482162475586
Scaler size (MB): 0.0007944107055664062


In [15]:
!pip install streamlit_jupyter

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 914.9/914.9 kB 3.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.2 MB 4.8 MB/s eta 0:00:01
   -------------------------------------- - 2.1/2.2 MB 3.9 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 3.6 MB/s eta 0:00:00


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [16]:
import streamlit as st
from streamlit_jupyter import StreamlitPatcher

StreamlitPatcher().jupyter()  # Register streamlit with jupyter-compatible wrappers


In [17]:
import streamlit as st
import numpy as np
import joblib
import json

# =====================================================
# Load model, scaler, feature schema
# =====================================================
model = joblib.load("house_price_model.pkl")
scaler = joblib.load("scaler.pkl")

with open("feature_names.json", "r") as f:
    feature_names = json.load(f)

# =====================================================
# Page Config
# =====================================================
st.set_page_config(
    page_title="House Price Prediction",
    page_icon="🏠",
    layout="centered"
)

# =====================================================
# UI
# =====================================================
st.title("🏠 House Price Prediction")
st.caption("Clean ML pipeline • Stable predictions")

st.sidebar.header("Enter House Details")

# EXACT features used in training
bedrooms = st.sidebar.number_input("Bedrooms", min_value=1, max_value=10, value=3)
bathrooms = st.sidebar.number_input("Bathrooms", min_value=1.0, max_value=5.0, value=2.0)
sqft_living = st.sidebar.number_input("Living Area (sqft)", min_value=300, max_value=10000, value=1500)
floors = st.sidebar.number_input("Floors", min_value=1.0, max_value=3.0, value=1.0)
grade = st.sidebar.slider("House Grade", min_value=1, max_value=13, value=7)

lat = st.sidebar.number_input("Latitude", value=47.6, format="%.4f")
long = st.sidebar.number_input("Longitude", value=-122.3, format="%.4f")

# =====================================================
# Build input feature vector (NO GUESSING)
# =====================================================
input_data = {
    "bedrooms": bedrooms,
    "bathrooms": bathrooms,
    "sqft_living": sqft_living,
    "floors": floors,
    "grade": grade,
    "lat": lat,
    "long": long
}

# Arrange features exactly as training order
feature_vector = [input_data[col] for col in feature_names]
features = np.array([feature_vector])

# =====================================================
# Prediction
# =====================================================
features_scaled = scaler.transform(features)
predicted_price = model.predict(features_scaled)[0]

# =====================================================
# Output
# =====================================================
st.subheader("💰 Predicted House Price")
st.success(f"USD {predicted_price:,.2f}")

# =====================================================
# Debug (optional – remove later)
# =====================================================
with st.expander("🔍 Debug Info"):
    st.write("Feature order:", feature_names)
    st.write("Input values:", feature_vector)


# 🏠 House Price Prediction

> Clean ML pipeline • Stable predictions

### 💰 Predicted House Price

>**expander starts**: 🔍 Debug Info

Feature order:

['bedrooms', 'bathrooms', 'sqft_living', 'floors', 'grade', 'lat', 'long']

Input values:

[3, 2.0, 1500, 1.0, 7, 47.6, -122.3]

>**expander ends**